# 01 E-Feature Extraction

Build an `EModelEFeatureExtractionScanConfig`, expand it via `GridScanGenerationTask`,
and register the `TaskConfig` entities in entitycore. The actual extraction runs on
**Fargate** via the obi-one service: `POST /declared/task/launch` with the
`task_type` and `config_id` starts a Fargate container that executes
`EModelEFeatureExtractionTask`, which downloads each `ElectricalCellRecording`'s
NWB asset from entitycore and extracts e-features through
[BluePyEModel](https://github.com/openbraininstitute/BluePyEModel)'s
`extract_save_features_protocols` (which wraps `bluepyefe` + eFEL). A minimal
`config/recipes.json` is written, but no `EModel_pipeline`, morphology or
mechanisms are needed at this stage.

**Reads from:** the entitycore staging project (`ElectricalCellRecording` entities).

**Writes to:** `obi-output/01_efeature_extraction/grid_scan/0/` — a working
directory containing the downloaded NWB files under `ephys_data/<entity-id>/`,
the minimal `config/recipes.json` and `config/extract_config/targets.json`, the
extraction `figures/`, and `extracted_features.json` (the serialised
`FitnessCalculatorConfiguration`). The optimisation stage in notebook 02 picks
the features file up and slots it into `config/features/<emodel>.json`.

## Imports

In [ ]:
from pathlib import Path

import obi_one as obi
from entitysdk import Client, ProjectContext
from entitysdk.models import TaskResult
from obi_auth import get_token
from obi_one.core.info import Info
from obi_one.scientific.tasks.emodel_building.task1_efeature_extraction.blocks.initialize import (
    ExtractionInitialize,
)
from obi_one.scientific.tasks.emodel_building.task1_efeature_extraction.blocks.settings import (
    Settings,
)


## Connect to entitycore staging

The extraction task downloads the recording NWB assets from entitycore. We use the
staging environment + test project bundled with `obi_one`. The first call to
`get_token(environment="staging")` opens a browser tab for authentication.


In [ ]:
token = get_token(environment="staging")
project_context = ProjectContext(
    virtual_lab_id=obi.LAB_ID_STAGING_TEST,
    project_id=obi.PROJECT_ID_STAGING_TEST,
)
db_client = Client(
    api_url="https://staging.cell-a.openbraininstitute.org/api/entitycore",
    project_context=project_context,
    token_manager=token,
)
print("Connected to entitycore staging.")

## Build the scan config

Every `bluepyefe` parameter relevant to extraction is exposed via blocks.
The default `efeatures_by_protocol.selection.protocols` is the L5PC mirror
(IDthresh/IDrest/IV/APWaveform/sAHP with a curated `extract=True` selection).
Step amplitudes and stimulus timing live in the NWB and are read by the task
at execution time, so neither needs to be set here.

Key settings:
- Rheobase is computed from the protocol marked with `is_rheobase_protocol=True`
  (IDthresh by default). The rheobase flag is set per-protocol.
- `validation_protocols` — protocols held out from optimisation (used only for validation).

In [ ]:
# A few arbitrary ElectricalCellRecording entities from the staging test project.
RECORDING_IDS = (
    # "492bdec5-2dce-4ae0-8b85-f020a1ad1d92",
    # "812a8721-1681-49a2-a155-59ab30981079",
    "00854004-4390-4a42-bf9d-5e672e8c8484",
    "66fd126c-f478-47e9-88ad-d12f7f27c84f"
)

scan_config = obi.EModelEFeatureExtractionScanConfig(
    info=Info(
        campaign_name="L5PC eFeature Extraction",
        campaign_description="Extract e-features from staging test recordings.",
    ),
    initialize=ExtractionInitialize(
        electrical_cell_recording=tuple(
            obi.ElectricalCellRecordingFromID(id_str=rid) for rid in RECORDING_IDS
        ),
    ),
    settings=Settings(
    ),
)
print("Settings:")
print(f"  default_std_value: {scan_config.settings.default_std_value}")
print()
print("efeatures_by_protocol:")
print(f"  threshold_based: {scan_config.efeatures_by_protocol.selection.threshold_based}")

## Inspect the recordings' protocols

Use the `/declared/electrical-cell-recording-protocols` endpoint to discover
the protocol (ecode) names stored in each recording's `stimuli` metadata.
The endpoint returns both per-recording and the union across all requested
recordings. We then use this union to populate
``scan_config.efeatures_by_protocol.selection.protocols`` instead of the L5PC defaults.

In [ ]:
import httpx

OBI_ONE_URL = "http://127.0.0.1:8100"

response = httpx.get(
    f"{OBI_ONE_URL}/declared/electrical-cell-recording-protocols",
    params={"recording_ids": list(RECORDING_IDS)},
    headers={"Authorization": f"Bearer {token}"},
    timeout=30,
)
response.raise_for_status()
data = response.json()

by_recording = data["by_recording"]
protocol_union = data["union"]

print("Per-recording protocols:")
for rid, prots in by_recording.items():
    print(f"  {rid}: {len(prots)} protocols")
print(f"\nUnion ({len(protocol_union)}): {protocol_union}")

## Drive `efeatures_by_protocol.selection.protocols` from the entity metadata

For every protocol name in the union, use `protocol_from_name()` to
instantiate the matching `Protocol` subclass, which already carries the eFEL
features valid for that protocol. Flip `extract=True` on every feature
so bluepyefe has targets to chew on. Protocols with no matching class
raise `KeyError` and are skipped.

Note: The protocol with `is_rheobase_protocol=True` is used for rheobase
estimation automatically — no separate setting is needed.

In [ ]:
from obi_one.scientific.tasks.emodel_building.task1_efeature_extraction.protocols_and_features.protocols import (
    protocol_from_name,
)

VALIDATION_PROTOCOLS = {"SpikeRec"}

configured_protocols = []
skipped_names = []
for name in protocol_union:
    try:
        instance = protocol_from_name(name)
    except KeyError:
        skipped_names.append(name)
        continue
    for feature in instance.features:
        feature.extract = True
    if name in VALIDATION_PROTOCOLS:
        instance.validation = True
    configured_protocols.append(instance)

scan_config.efeatures_by_protocol.selection.protocols = tuple(configured_protocols)

print(f"Matched {len(configured_protocols)} protocols from the union")
if skipped_names:
    print(f"Skipped {len(skipped_names)} (no matching eCode): {skipped_names}")
for p in scan_config.efeatures_by_protocol.selection.protocols:
    tag = " [validation]" if p.validation else ""
    print(f"  {p.name}: {len(p.selected_efeatures())} features extracting{tag}")

## Sanity-check the amplitude reader

The task auto-discovers per-protocol step amplitudes from each NWB so the
user never has to type them. Download one recording's NWB, run the reader
against the matched protocols, and confirm the values look reasonable
(BBP NWBs typically expose currents in nA, so step amplitudes for IV are
around -0.1–0 nA, APWaveform around 0.2–0.3 nA, etc.).


In [ ]:
import tempfile
from pathlib import Path

from obi_one.scientific.library.electrical_cell_recording_properties import (
    read_amplitudes_from_nwb,
)

matched_names = [p.name for p in scan_config.efeatures_by_protocol.selection.protocols]

sample_recording = obi.ElectricalCellRecordingFromID(id_str=RECORDING_IDS[0])
with tempfile.TemporaryDirectory() as tmp:
    nwb_path = sample_recording.download_asset(dest_dir=Path(tmp), db_client=db_client)
    amps = read_amplitudes_from_nwb(nwb_path, matched_names)

print(f"Discovered amplitudes (nA) in {sample_recording.id_str}:")
for proto, amp_list in amps.items():
    print(f"  {proto}: {amp_list}")

## Run the grid scan

Per-coordinate output goes to `obi-output/01_efeature_extraction/grid_scan/<idx>/`
(the `ZERO_INDEX` directory option keeps the path stable for downstream notebooks).


In [ ]:
grid_scan = obi.GridScanGenerationTask(
    form=scan_config,
    output_root="../../../obi-output/01_efeature_extraction/grid_scan",
    coordinate_directory_option="ZERO_INDEX",
)
grid_scan.execute(db_client=db_client)

## Inspect the extracted features

In [ ]:
import json
from pathlib import Path

coord_root = Path(grid_scan.single_configs[0].coordinate_output_root).resolve()
print("Coordinate output:", coord_root)
print()

features_path = coord_root / "extracted_features.json"
print("Features:", features_path)
features = json.loads(features_path.read_text())
print("Top-level keys:", list(features))
print(
    f"Extracted {len(features.get('efeatures', []))} efeatures across"
    f" {len(features.get('protocols', []))} protocols."
)

## Inspect the recipe (validation_protocols)

The extraction task writes a `recipes.json` that carries `validation_protocols`
forward to Workflow A. Protocols marked `validation=True` in the config are
collected and listed here. Confirm they're present.

In [ ]:
recipes_path = coord_root / "config" / "recipes.json"
recipes = json.loads(recipes_path.read_text())
ps = recipes["emodel"]["pipeline_settings"]
print("Pipeline settings in recipe:")
print(f"  validation_protocols: {ps.get('validation_protocols')}")
print(f"  extract_absolute_amplitudes: {ps.get('extract_absolute_amplitudes')}")
print(f"  efel_settings: {ps.get('efel_settings')}")
print(f"  name_Rin_protocol: {ps.get('name_Rin_protocol')}")
print(f"  name_rmp_protocol: {ps.get('name_rmp_protocol')}")


## Registered config entities & launch on Fargate

The `GridScanGenerationTask.execute()` call registered two `TaskConfig` entities
(campaign + single) in entitycore. The actual task execution (feature extraction,
entity registration) runs on **Fargate**, handled entirely by the obi-one service:

1. **Submit the job** — `POST /declared/task/launch` with `task_type` + `config_id`
2. **obi-one service** creates `TaskActivity`, submits job to launch system
3. **Fargate container** runs `EModelEFeatureExtractionTask.execute()`
4. The task registers a `TaskResult` entity and updates the `TaskActivity` with `generated_ids`

After the job completes, query the `TaskActivity` for `generated_ids` to find the
`TaskResult` ID. Copy that ID into notebook `02_emodel_optimization.ipynb` as the
extraction `TaskResultFromID` input.

In [ ]:
import httpx

# Campaign TaskConfig — stored on the scan config (grid_scan.form)
campaign = grid_scan.form.campaign
campaign_id = campaign.id if campaign else None

# Single TaskConfig — stored on the first single coordinate config
single = grid_scan.single_configs[0].single_entity
single_id = single.id if single else None

print("=== Registered Config Entities ===")
print(f"  Campaign TaskConfig: {campaign_id}")
print(f"  Single TaskConfig:   {single_id}")
print()

# Launch the task on Fargate via the obi-one service.
# Requires the obi-one API server running on OBI_ONE_URL.
# The /declared/task/launch endpoint requires project-context headers.
response = httpx.post(
    f"{OBI_ONE_URL}/declared/task/launch",
    json={
        "task_type": "efeature_extraction",
        "config_id": str(single_id),
    },
    headers={
        "Authorization": f"Bearer {token}",
        "virtual-lab-id": str(obi.LAB_ID_STAGING_TEST),
        "project-id": str(obi.PROJECT_ID_STAGING_TEST),
    },
    timeout=30,
)
response.raise_for_status()
launch_info = response.json()
print(f"Job launched: {launch_info}")
print(f"  activity_id: {launch_info['activity_id']}")
print(f"  job_id:      {launch_info['job_id']}")

# After the Fargate job completes, query the TaskActivity for generated_ids:
activity = db_client.get_entity(
    entity_id=launch_info["activity_id"],
    entity_type="TaskActivity",  # ty:ignore[invalid-argument-type]
)
generated_ids = activity.generated_ids
print(f"Generated IDs: {generated_ids}")
print()
print("Copy the TaskResult ID into notebook 02:")
print(f'  TaskResultFromID(id_str="<task_result_id_from_generated_ids>")')